# Prepare Batch Schedules

Builds the shared schedules that the Claude, Gemini and OpenAI inference notebooks send as requests, saved in `data/schedules/`:

- `rating_schedule-{all_tasks,1000_tasks}.parquet`: one row per screen task and main rating aspect (and trial when `NUM_TRIALS > 1`)
- `critiques_schedule-{all_tasks,1000_tasks}.parquet`: one row per screen task
- `critiques_schedule-all_tasks_excluding_1000.parquet`: critique tasks outside the 1000-task subset

The 1000-task subset is the first task of each of the first 1000 screens. The ratings section also builds the few-shot
samples file (the 3 lowest-, 1 middle- and 3 highest-rated tasks by total rating).

# Imports

In [ ]:
import sys
import json
from pathlib import Path
import pandas as pd
import os

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "utils" / "paths.py").exists())
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from utils.inference_utils import EVALUATION_MAIN_ASPECTS, EVALUATION_FIVE_ASPECTS
from utils.data_setup import get_dataset_file_path
from utils.paths import SCHEDULES_DIR

# Config and Constants

In [ ]:
NUM_TRIALS = 1

# Data Loading

In [ ]:
# Both stages schedule the same tasks; the ratings paths also give the few-shot samples file.
uicrit_file, base64_screens_file, few_shot_samples_file = get_dataset_file_path("ratings")

uicrit_df = pd.read_parquet(uicrit_file)
screenshots_df = pd.read_parquet(base64_screens_file)

print("UICrit rows:", len(uicrit_df))
print("Screens rows:", len(screenshots_df))

In [ ]:
schedule_dir = SCHEDULES_DIR
schedule_dir.mkdir(parents=True, exist_ok=True)

rating_schedule_file_1000_tasks = schedule_dir / "rating_schedule-1000_tasks.parquet"
rating_schedule_file_all_tasks = schedule_dir / "rating_schedule-all_tasks.parquet"
critiques_schedule_file_1000_tasks = schedule_dir / "critiques_schedule-1000_tasks.parquet"
critiques_schedule_file_all_tasks = schedule_dir / "critiques_schedule-all_tasks.parquet"
critiques_schedule_file_all_tasks_excluding_1000 = schedule_dir / "critiques_schedule-all_tasks_excluding_1000.parquet"

# Preparing Schedules

Each cell builds its file only when it is missing; delete a file to rebuild it.

## Ratings Schedule

In [ ]:
# Build ratings schedule for all tasks
if rating_schedule_file_all_tasks.exists():
    print("Already exists (delete it to rebuild):", rating_schedule_file_all_tasks)
else:
    records = []
    for _, row in uicrit_df.iterrows():
        screen_task_id = row["screen_task_id"]
        for trial in range(1, NUM_TRIALS + 1):
            for aspect in EVALUATION_MAIN_ASPECTS:
                record = {
                    "custom_id": (
                        f"{screen_task_id}&trial{trial}&aspect={aspect}"
                        if NUM_TRIALS > 1
                        else f"{screen_task_id}&aspect={aspect}"
                    ),
                    "screen_task_id": screen_task_id,
                    "aspect": aspect,
                }
                if NUM_TRIALS > 1:
                    record["trial"] = trial
                records.append(record)

    trials_schedule_df = pd.DataFrame.from_records(records)
    trials_schedule_df.to_parquet(rating_schedule_file_all_tasks, index=False)

    print(trials_schedule_df.shape)
    trials_schedule_df.head(4)

In [ ]:
# Build ratings schedule for the first task of 1000 screens
if rating_schedule_file_1000_tasks.exists():
    print("Already exists (delete it to rebuild):", rating_schedule_file_1000_tasks)
else:
    trials_schedule_df = pd.read_parquet(rating_schedule_file_all_tasks)
    first_tasks_df = uicrit_df.drop_duplicates(subset="screen_id", keep="first")
    selected_task_ids = first_tasks_df["screen_task_id"].head(1000).unique()

    trials_schedule_df = trials_schedule_df[trials_schedule_df["screen_task_id"].isin(selected_task_ids)].copy()
    trials_schedule_df.to_parquet(rating_schedule_file_1000_tasks, index=False)

    print(trials_schedule_df.shape)
    trials_schedule_df.head(4)

In [ ]:
# Optional: rebuild few-shot samples file for ratings
if few_shot_samples_file.exists():
    print("Already exists (delete it to rebuild):", few_shot_samples_file)
else:
    few_shot_samples_file.parent.mkdir(parents=True, exist_ok=True)
    uicrit_df_ratings = uicrit_df.copy()
    uicrit_df_ratings["total_rating"] = uicrit_df_ratings[EVALUATION_FIVE_ASPECTS].sum(axis=1)
    uicrit_df_ratings = uicrit_df_ratings.sort_values(by="total_rating")

    lowest_rated = uicrit_df_ratings.head(3).drop(columns=["total_rating", "comments_source", "comments"], errors="ignore")
    mid_rated = uicrit_df_ratings.iloc[[len(uicrit_df_ratings) // 2]].drop(columns=["total_rating", "comments_source", "comments"], errors="ignore")
    highest_rated = uicrit_df_ratings.tail(3).drop(columns=["total_rating", "comments_source", "comments"], errors="ignore")

    few_shot_samples_df = pd.concat([lowest_rated, mid_rated, highest_rated]).merge(screenshots_df, on="screen_id", how="left")
    few_shot_samples_df.to_parquet(few_shot_samples_file, index=False)

    few_shot_samples_df.head()

## Critiques Schedule

In [ ]:
# Build critiques schedule for all tasks
if critiques_schedule_file_all_tasks.exists():
    print("Already exists (delete it to rebuild):", critiques_schedule_file_all_tasks)
else:
    records = []
    for _, row in uicrit_df.iterrows():
        screen_task_id = row["screen_task_id"]
        records.append({
            "custom_id": screen_task_id,
            "screen_task_id": screen_task_id,
        })

    critiques_schedule_df = pd.DataFrame.from_records(records)
    critiques_schedule_df.to_parquet(critiques_schedule_file_all_tasks, index=False)

    print(critiques_schedule_df.shape)
    critiques_schedule_df.head(4)

In [ ]:
# Build critiques schedule for the first task of 1000 screens
if critiques_schedule_file_1000_tasks.exists():
    print("Already exists (delete it to rebuild):", critiques_schedule_file_1000_tasks)
else:
    critiques_schedule_df = pd.read_parquet(critiques_schedule_file_all_tasks)
    first_tasks_df = uicrit_df.drop_duplicates(subset="screen_id", keep="first")
    selected_task_ids = first_tasks_df["screen_task_id"].head(1000).unique()

    critiques_schedule_df = critiques_schedule_df[critiques_schedule_df["screen_task_id"].isin(selected_task_ids)].copy()
    critiques_schedule_df.to_parquet(critiques_schedule_file_1000_tasks, index=False)

    print(critiques_schedule_df.shape)
    critiques_schedule_df.head(4)

In [ ]:
# Optional: build critiques schedule for all_tasks excluding the 1000-task subset
if critiques_schedule_file_all_tasks_excluding_1000.exists():
    print("Already exists (delete it to rebuild):", critiques_schedule_file_all_tasks_excluding_1000)
else:
    schedule_1000_tasks_df = pd.read_parquet(critiques_schedule_file_1000_tasks)
    all_tasks_schedule_df = pd.read_parquet(critiques_schedule_file_all_tasks)

    schedule_all_tasks_excluding_1000_df = all_tasks_schedule_df[
        ~all_tasks_schedule_df["screen_task_id"].isin(schedule_1000_tasks_df["screen_task_id"])
    ].copy()

    schedule_all_tasks_excluding_1000_df.to_parquet(critiques_schedule_file_all_tasks_excluding_1000, index=False)

    print(schedule_all_tasks_excluding_1000_df.shape)
    schedule_all_tasks_excluding_1000_df.head(4)